# Chapter 18 — Serving at Scale

Every number in Chapter 18, recomputed, plus small simulations of the three systems ideas:
PagedAttention's allocator, continuous batching, and speculative decoding's exactness guarantee.
Each cell **asserts** what it checks, so `python tests/run_notebooks.py 18` fails if the chapter and
the code disagree. Speculative decoding runs on two toy bigram "models" instead of GPT-2 small and
large (Exercise 18.1's analogue: the guarantee is about distributions, so any pair of models tests it).

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 18.1 — contiguous reservation wastes what the sequence never uses

In [ ]:
print(f"200 of 4096 reserved tokens used: {1 - 200 / 4096:.1%} wasted")
assert round(100 * (1 - 200 / 4096)) == 95

## Exercise 18.2 — a block allocator, simulated on 1000 lognormal requests

`BlockManager` with `allocate`, `append`, `free` and copy-on-write `fork`. Request lengths are
lognormal with mean 300, capped at 4096; contiguous allocation reserves 4096 per request.

**DISCREPANCY.** The chapter expects contiguous allocation to waste "70–90%" with these lengths.
Reserving 4096 for sequences averaging ~300 tokens wastes 1 − 300/4096 ≈ **93%** — above the stated
range by construction (the range would fit a reservation of ~1000–3000, or a mean of ~400–1200).
Paged allocation wastes under 3%, as the chapter says.

In [ ]:
class BlockManager:
    def __init__(self, total_blocks, block_size=16):
        self.bs, self.free_list = block_size, list(range(total_blocks))
        self.ref = {}                    # physical block -> reference count
        self.tables, self.lens = {}, {}  # seq -> [physical blocks], seq -> tokens
    def _new(self):
        b = self.free_list.pop(); self.ref[b] = 1; return b
    def allocate(self, seq, n_tokens):
        self.tables[seq] = [self._new() for _ in range(-(-n_tokens // self.bs))]
        self.lens[seq] = n_tokens
    def append(self, seq):
        t = self.tables[seq]
        if self.lens[seq] % self.bs == 0:
            t.append(self._new())                      # need a fresh block
        elif self.ref[t[-1]] > 1:                      # copy-on-write: last block is shared
            self.ref[t[-1]] -= 1; t[-1] = self._new()
        self.lens[seq] += 1
    def fork(self, parent, child):
        self.tables[child] = list(self.tables[parent]); self.lens[child] = self.lens[parent]
        for b in self.tables[child]: self.ref[b] += 1
    def free(self, seq):
        for b in self.tables.pop(seq):
            self.ref[b] -= 1
            if self.ref[b] == 0: del self.ref[b]; self.free_list.append(b)
        del self.lens[seq]
    def used_blocks(self):
        return len(self.ref)

bm = BlockManager(100)
bm.allocate("a", 40)                                   # 3 blocks
bm.fork("a", "b")
assert bm.used_blocks() == 3                           # a shared prefix costs nothing extra
bm.append("b")                                         # b writes into the shared last block -> copy
assert bm.used_blocks() == 4 and bm.tables["a"][:2] == bm.tables["b"][:2]
bm.free("a"); bm.free("b")
assert bm.used_blocks() == 0 and len(bm.free_list) == 100

torch.manual_seed(0)
sigma = 1.0
mu = math.log(300) - sigma ** 2 / 2                    # lognormal with mean 300
lens = torch.distributions.LogNormal(mu, sigma).sample((1000,)).round().clamp(1, 4096).long()
bm = BlockManager(10**6)
for i, n in enumerate(lens.tolist()):
    bm.allocate(i, 1)
    for _ in range(n - 1): bm.append(i)               # grow one token at a time, on demand
used = lens.sum().item()
paged_waste = 1 - used / (bm.used_blocks() * 16)
contig_waste = 1 - used / (4096 * len(lens))
print(f"mean length {lens.float().mean():.0f}; contiguous wastes {contig_waste:.1%}, paged wastes {paged_waste:.1%}")
assert paged_waste < 0.03
assert contig_waste > 0.90                             # chapter says 70-90%

## 18.2 — static vs continuous batching, counted in slot-steps

8 slots, 64 requests with skewed lengths. Static batching holds a batch until its longest member
finishes; continuous batching refills a slot the step it frees.

In [ ]:
torch.manual_seed(1)
req = torch.distributions.LogNormal(math.log(100), 1.2).sample((64,)).round().clamp(1, 4000).long().tolist()
slots = 8
static_steps = sum(max(req[i:i + slots]) for i in range(0, len(req), slots))
import heapq
finish = [0] * slots
for n in req:                                          # each request takes the earliest free slot
    t = heapq.heappop(finish) if len(finish) == slots else 0
    heapq.heappush(finish, t + n)
continuous_steps = max(finish)
print(f"static {static_steps} steps, continuous {continuous_steps} steps: {static_steps / continuous_steps:.1f}x")
assert static_steps / continuous_steps > 2                  # the chapter: typically 2-4x
assert 32 - 4 == 28                                         # seq A in the diagram: done at 4, idle 28

## 18.3 — the worked example: how many drafted tokens survive (α = 0.7, k = 4)

In [ ]:
def expected_tokens(alpha, k):
    return (1 - alpha ** (k + 1)) / (1 - alpha)
a, k = 0.7, 4
dist = [a ** i * (1 - a) for i in range(k)] + [a ** k]
print("accept exactly 0..4:", [round(p, 4) for p in dist])
assert [round(p, 4) for p in dist] == [0.3, 0.21, 0.147, 0.1029, 0.2401]
assert abs(sum(dist) - 1) < 1e-12
E = sum(i * p for i, p in enumerate(dist)) + 1             # + the corrected / bonus token
assert abs(E - expected_tokens(a, k)) < 1e-12 and round(E, 2) == 2.77
table = {0.3: (1.39, 1.43, 1.43), 0.5: (1.75, 1.94, 2.00), 0.7: (2.19, 2.77, 3.20), 0.9: (2.71, 4.10, 6.13)}
for al, row in table.items():
    got = tuple(round(expected_tokens(al, kk), 2) for kk in (2, 4, 8))
    print(f"alpha {al}: k=2,4,8 -> {got}")
    assert got == row
assert round(expected_tokens(0.7, 4), 2) == 2.77 and round(expected_tokens(0.9, 8), 2) == 6.13
assert expected_tokens(0.3, 8) - expected_tokens(0.3, 4) < 0.01

## Exercise 18.1 — speculative decoding is exact (verified with a chi-squared test)

Two random bigram models over 6 tokens: a target p(·|prev) and a deliberately different draft
q(·|prev). Accept a drafted token with probability min(1, p/q); on rejection sample from
normalise(max(0, p − q)); if all k survive, take a bonus token from p. Compare the next-token
distribution after a fixed context with plain sampling from the target.

In [ ]:
torch.manual_seed(0)
V = 6
Pt = (torch.randn(V, V) * 1.5).softmax(-1)               # target: row = previous token
Qd = (torch.randn(V, V) * 1.5).softmax(-1)               # draft
def spec_step(prev, k=4):
    out, cur, drafts, qs = [], prev, [], []
    for _ in range(k):                                    # 1. draft k tokens
        t = torch.multinomial(Qd[cur], 1).item(); drafts.append(t); qs.append(Qd[cur]); cur = t
    ctx = [prev] + drafts
    accepted = 0
    for i, t in enumerate(drafts):                        # 2-3. verify left to right
        p_i = Pt[ctx[i]]
        if torch.rand(1).item() < min(1.0, (p_i[t] / qs[i][t]).item()):
            out.append(t); accepted += 1
        else:
            resid = (p_i - qs[i]).clamp(min=0)
            out.append(torch.multinomial(resid / resid.sum(), 1).item())
            return out, accepted
    out.append(torch.multinomial(Pt[drafts[-1]], 1).item())   # bonus token
    return out, accepted
N = 20_000
firsts, acc = torch.zeros(V), 0
for _ in range(N):
    o, a_ = spec_step(0)
    firsts[o[0]] += 1; acc += a_
expected = Pt[0] * N
chi2 = ((firsts - expected) ** 2 / expected).sum().item()
print(f"chi-squared = {chi2:.2f} on 5 dof (p = 0.001 critical value 20.5)")
assert chi2 < 20.5                                       # indistinguishable from the target
naive = ((Qd[0] * N - expected) ** 2 / expected).sum().item()
assert naive > 1000                                      # while the draft itself is far off
alpha_theory = torch.minimum(Pt, Qd).sum(-1)             # P(accept) per context = sum min(p, q)
print(f"acceptance from context 0: theory {alpha_theory[0]:.3f}")

## 18.4 — why weight quantization speeds up decode

In [ ]:
P = 70e9
bf16, int4 = 2 * P, 0.5 * P
print(f"70B: {bf16 / 1e9:.0f} GB -> {int4 / 1e9:.0f} GB per forward; decode bound improves {bf16 / int4:.0f}x")
assert (bf16 / 1e9, int4 / 1e9) == (140, 35) and bf16 / int4 == 4

## Quiz — 1000 users sharing a 2000-token system prompt: prefix caching

In [ ]:
without, with_cache = 1000 * 2000, 2000
print(f"prefill tokens: {without:,} without prefix caching, {with_cache:,} with it")
assert without / with_cache == 1000